In [42]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

In [43]:
url = "https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks"
table_attribs = [
    'Name',
    'MC_USD_Billion',
    'MC_GBP_Billion',
    'MC_EUR_Billion',
    'MC_INR_Billion'
]

table_name =  'Largest_banks'
exchange_rate_csv = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-PY0221EN-Coursera/labs/v2/exchange_rate.csv'
output_csv = './Largest_banks_data.csv'
 
log_file = 'code_log.txt'

df = pd.DataFrame(columns=table_attribs)

In [44]:
# --- Task 1: Logging Function ---
def log_progress(message):
    ''' Logs the mentioned message at a given stage of the code execution to a log file. '''
    
    timestamp_format = '%Y-%b-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second
    now = datetime.now()
    timestamp = now.strftime(timestamp_format)
    with open(log_file, "a") as f:
        f.write(f"{timestamp} : {message}\n")

In [45]:
def extract(url, table_attribs):
    page = requests.get(url).text
    data = BeautifulSoup(page, 'html.parser')
    
    rows_list = []
    tables = data.find_all('tbody')
    rows = tables[0].find_all('tr')

    for row in rows:
        col = row.find_all('td')
        if len(col) != 0:
            bank_name = col[1].text.strip()
            market_cap = float(col[2].text.strip().replace(',', '').replace('\n', ''))
            rows_list.append({'Name': bank_name, 'MC_USD_Billion': market_cap})

    df = pd.DataFrame(rows_list)
    return df

In [46]:
extract(url, table_attribs)

,Name,MC_USD_Billion
0,JPMorgan Chase,432.92
1,Bank of America,231.52
2,Industrial and Commercial Bank of China,194.56
3,Agricultural Bank of China,160.68
4,HDFC Bank,157.91
5,Wells Fargo,155.87
6,HSBC Holdings PLC,148.90
7,Morgan Stanley,140.83
8,China Construction Bank,139.82
9,Bank of China,136.81


In [47]:
df = extract(url, table_attribs)

In [48]:
# --- Task 3: Transformation Function ---
def transform(df, csv_path):
    ''' Reads exchange rate CSV, converts Market Cap to GBP, EUR, and INR, and rounds to 2 decimal places. '''
    exchange_rate_df = pd.read_csv(csv_path)
    dict_rates = exchange_rate_df.set_index('Currency').to_dict()['Rate']
    
    # تحويل القيم والتقريب لرقيمين عشريين
    df['MC_GBP_Billion'] = [np.round(x * dict_rates['GBP'], 2) for x in df['MC_USD_Billion']]
    df['MC_EUR_Billion'] = [np.round(x * dict_rates['EUR'], 2) for x in df['MC_USD_Billion']]
    df['MC_INR_Billion'] = [np.round(x * dict_rates['INR'], 2) for x in df['MC_USD_Billion']]
    
    return df

In [49]:
transform(df, exchange_rate_csv)

,Name,MC_USD_Billion,MC_GBP_Billion,MC_EUR_Billion,MC_INR_Billion
0,JPMorgan Chase,432.92,346.34,402.62,35910.71
1,Bank of America,231.52,185.22,215.31,19204.58
2,Industrial and Commercial Bank of China,194.56,155.65,180.94,16138.75
3,Agricultural Bank of China,160.68,128.54,149.43,13328.41
4,HDFC Bank,157.91,126.33,146.86,13098.63
5,Wells Fargo,155.87,124.70,144.96,12929.42
6,HSBC Holdings PLC,148.90,119.12,138.48,12351.26
7,Morgan Stanley,140.83,112.66,130.97,11681.85
8,China Construction Bank,139.82,111.86,130.03,11598.07
9,Bank of China,136.81,109.45,127.23,11348.39


In [50]:
df = transform(df, exchange_rate_csv)

In [51]:
# Task 4: Load to CSV
def load_to_csv(df, csv_path):
    df.to_csv(csv_path, index=False)

In [52]:
load_to_csv(df, output_csv)

In [53]:
# Task 5: Load to DB
def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

In [54]:
load_to_db(df, sqlite3.connect('banks_data.db'), table_name)

In [55]:
# Task 6: Run Query
def run_query(query_statement, sql_connection): # ✅ تصحيح run_query
    print(f'query : {query_statement}\n')
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)

In [56]:
run_query("SELECT * FROM Largest_banks", sqlite3.connect('banks_data.db'))

query : SELECT * FROM Largest_banks

                                      Name  MC_USD_Billion  MC_GBP_Billion  \
0                           JPMorgan Chase          432.92          346.34   
1                          Bank of America          231.52          185.22   
2  Industrial and Commercial Bank of China          194.56          155.65   
3               Agricultural Bank of China          160.68          128.54   
4                                HDFC Bank          157.91          126.33   
5                              Wells Fargo          155.87          124.70   
6                        HSBC Holdings PLC          148.90          119.12   
7                           Morgan Stanley          140.83          112.66   
8                  China Construction Bank          139.82          111.86   
9                            Bank of China          136.81          109.45   

   MC_EUR_Billion  MC_INR_Billion  
0          402.62        35910.71  
1          215.31        19204.5

In [57]:

log_progress('ETL Process Started')
log_progress('Data extraction completed')
log_progress('Data transformation completed')
log_progress('Data saved to CSV file')
log_progress('SQL Connection initiated')
log_progress('Data loaded to Database as table. Running queries now')
log_progress('Process Completed')

In [58]:

# Task 7: Verify Log File
def verify_log_file(log_file):
    with open(log_file, 'r') as file:
        print(file.read())

verify_log_file('code_log.txt')

2026-Oct-01-01:40:08 : Data extraction, transformation, and loading completed successfully.
2026-Oct-01-01:40:16 : Data extraction, transformation, and loading completed successfully.
2026-Oct-01-01:41:52 : ETL Process Started
2026-Oct-01-01:41:52 : Data extraction completed
2026-Oct-01-01:41:52 : Data transformation completed
2026-Oct-01-01:41:52 : Data saved to CSV file
2026-Oct-01-01:41:52 : SQL Connection initiated
2026-Oct-01-01:41:52 : Data loaded to Database as table. Running queries now
2026-Oct-01-01:41:52 : Process Completed
2026-Oct-01-01:42:08 : ETL Process Started
2026-Oct-01-01:42:08 : Data extraction completed
2026-Oct-01-01:42:08 : Data transformation completed
2026-Oct-01-01:42:08 : Data saved to CSV file
2026-Oct-01-01:42:08 : SQL Connection initiated
2026-Oct-01-01:42:08 : Data loaded to Database as table. Running queries now
2026-Oct-01-01:42:08 : Process Completed

